In [1]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt #为了画图
%matplotlib inline

In [2]:
words=open('names.txt', 'r').read().splitlines()
words[:8]

['emma', 'olivia', 'ava', 'isabella', 'sophia', 'charlotte', 'mia', 'amelia']

In [3]:
len(words)

32033

In [4]:
chars=sorted(list(set(''.join(words))))
stoi={s:i+1 for i,s in enumerate(chars)}
stoi['.']=0
itos={i:s for s,i in stoi.items()}
print(itos)

{1: 'a', 2: 'b', 3: 'c', 4: 'd', 5: 'e', 6: 'f', 7: 'g', 8: 'h', 9: 'i', 10: 'j', 11: 'k', 12: 'l', 13: 'm', 14: 'n', 15: 'o', 16: 'p', 17: 'q', 18: 'r', 19: 's', 20: 't', 21: 'u', 22: 'v', 23: 'w', 24: 'x', 25: 'y', 26: 'z', 0: '.'}


In [32]:
# 建立训练数据
block_size=3 #上下文长度，也就是模型一次能看到几个字符
X,Y=[],[]
for w in words[:5]: #只用前1000个单词来训练
    #print(w)
    context=[0]*block_size #用0来表示context的开始
    for ch in w+'.':
        ix=stoi[ch]
        X.append(context)
        Y.append(ix)
        #print(''.join(itos[i] for i in context), '---->', itos[ix])
        context=context[1:]+[ix] #把context向右移动，加入新的字符


X=torch.tensor(X)
Y=torch.tensor(Y)


In [33]:
X.shape, X.dtype, Y.shape, Y.dtype

(torch.Size([32, 3]), torch.int64, torch.Size([32]), torch.int64)

In [7]:
C=torch.randn(27,2)

In [8]:
emb=C[X]
emb.shape

torch.Size([32, 3, 2])

In [9]:
w1=torch.randn((6,100))
b1=torch.randn(100)

In [10]:
torch.cat([emb[:,0,:],emb[:,1,:],emb[:,2,:]],1)

tensor([[ 4.5791e-01,  2.3727e-01,  4.5791e-01,  2.3727e-01,  4.5791e-01,
          2.3727e-01],
        [ 4.5791e-01,  2.3727e-01,  4.5791e-01,  2.3727e-01,  8.6175e-01,
          1.3438e+00],
        [ 4.5791e-01,  2.3727e-01,  8.6175e-01,  1.3438e+00,  3.5878e-01,
          1.1041e+00],
        [ 8.6175e-01,  1.3438e+00,  3.5878e-01,  1.1041e+00,  3.5878e-01,
          1.1041e+00],
        [ 3.5878e-01,  1.1041e+00,  3.5878e-01,  1.1041e+00, -5.6589e-01,
          1.4282e+00],
        [ 4.5791e-01,  2.3727e-01,  4.5791e-01,  2.3727e-01,  4.5791e-01,
          2.3727e-01],
        [ 4.5791e-01,  2.3727e-01,  4.5791e-01,  2.3727e-01, -7.6851e-01,
         -8.0604e-02],
        [ 4.5791e-01,  2.3727e-01, -7.6851e-01, -8.0604e-02, -1.4359e+00,
         -1.9297e-01],
        [-7.6851e-01, -8.0604e-02, -1.4359e+00, -1.9297e-01, -2.2428e-01,
          1.5228e-03],
        [-1.4359e+00, -1.9297e-01, -2.2428e-01,  1.5228e-03, -6.6078e-01,
          8.8986e-01],
        [-2.2428e-01,  1.5228e

In [11]:
torch.cat(torch.unbind(emb,1),dim=1).shape

torch.Size([32, 6])

torch.cat(torch.unbind(emb,1),dim=1).shape
emb.view(-1,6)
这两个方法一样，但是优先考虑下面的，因为上面的会重新分配内存

In [12]:
h=torch.tanh(emb.view(-1,6) @ w1 + b1)
h.shape

torch.Size([32, 100])

上面有广播机制，
32,100
  ，100
这两个张量相加，会将下面的变成（1，100），

In [13]:
w2=torch.randn((100,27))
b2=torch.randn(27)
logits=h @ w2 + b2
logits.shape

torch.Size([32, 27])

In [14]:
counts=logits.exp()
prob=counts/counts.sum(1,keepdim=True)# 按照第一维度来归一化，也是一个softmax

In [15]:
loss=-prob[torch.arange(32),Y].log().mean()
loss

tensor(18.4632)

In [68]:
# 建立训练数据
block_size=3 #上下文长度，也就是模型一次能看到几个字符
X,Y=[],[]
for w in words: #只用前1000个单词来训练
    #print(w)
    context=[0]*block_size #用0来表示context的开始
    for ch in w+'.':
        ix=stoi[ch]
        X.append(context)
        Y.append(ix)
        #print(''.join(itos[i] for i in context), '---->', itos[ix])
        context=context[1:]+[ix] #把context向右移动，加入新的字符


X=torch.tensor(X)
Y=torch.tensor(Y)


In [69]:
## -------------------  完整步骤 -------------------
X.shape, X.dtype, Y.shape, Y.dtype

(torch.Size([228146, 3]), torch.int64, torch.Size([228146]), torch.int64)

In [71]:
g=torch.Generator().manual_seed(2147483647)
C=torch.randn(27,2, generator=g)
w1=torch.randn((6,100), generator=g)
b1=torch.randn(100, generator=g)
w2=torch.randn((100,27), generator=g)
b2=torch.randn(27, generator=g)
parameters=[C,w1,b1,w2,b2]

In [73]:
sum(p.nelement() for p in parameters)

3481

In [75]:
for p in parameters:
    p.requires_grad=True

In [77]:
lre=torch.linspace(-3,0,1000)
lrs=10**lre #让变化率随指数变化


In [101]:
# lri=[]
# lossi=[]
for i in range(1000):

    #构建 mini batch
    ix=torch.randint(0,X.shape[0],(32,))
    # forward pass
    emb=C[X[ix]]  #(32,3,2)
    h=torch.tanh(emb.view(-1,6) @ w1 + b1) #(32,100)
    logits=h @ w2 + b2 #(32,27)
    # counts=logits.exp()
    # prob=counts/counts.sum(1,keepdim=True)
    # loss=-prob[torch.arange(32),Y].log().mean()
    loss=F.cross_entropy(logits, Y[ix])
    

    #backward pass
    for p in parameters:
        p.grad=None
    loss.backward()
    #update
    # lr=lrs[i]
    lr=0.1
    for p in parameters:
        p.data +=  -lr* p.grad
    #track stats
    # lri.append(lr)
    # lossi.append(loss.item())

print(loss.item())

2.4846649169921875


In [102]:
emb=C[X]
h=torch.tanh(emb.view(-1,6) @ w1 + b1)
logits=h @ w2 + b2
loss=F.cross_entropy(logits, Y)
print(loss)

tensor(2.4092, grad_fn=<NllLossBackward0>)


In [44]:
torch.randint(0,X.shape[0],(32,))

tensor([ 61949, 137164,  95317,  86136, 159055, 199503, 134894, 171084, 123397,
         44985, 164456, 121674,   9960,  13538, 133461,  20981,  57389,  83275,
          3353, 110330,  21281, 210878, 210961, 157724,  39597,  87813, 178719,
        212482,  40107,  90565, 162269, 161298])

# counts=logits.exp()
# prob=counts/counts.sum(1,keepdim=True)
# loss=-prob[torch.arange(32),Y].log().mean()
上下两种方法等价，交叉熵损失函数，
1.下面的pytorch会更加快速，因为不需要向上面一样建立那么多中间的张量
2.下面的反向传播会更加快速
3.有些非常大的数不能用上面的表示，具体看下面的例子
# F.cross_entropy(logits, Y)

In [21]:
# this is an example
l=torch.tensor([-5, -3, 0, 3])
counts=l.exp()
prob=counts/counts.sum()
prob

tensor([3.1870e-04, 2.3549e-03, 4.7299e-02, 9.5003e-01])

tensor([3.1870e-04, 2.3549e-03, 4.7299e-02, 9.5003e-01])

In [22]:
# this is an example
l=torch.tensor([-100, -3, 0, 100])
counts=l.exp()
prob=counts/counts.sum()
prob

tensor([0., 0., 0., nan])

tensor([0., 0., 0., nan])
原因：向量经过 exp() 后包含了一个溢出的无穷大（inf），于是归一化时产生了 0 和 nan。
pytorch的解决方法是：将tensor数组所有元素都减去最大值，因为是指数运算，同时加或减相当于在后续的除法中上下同时除以一个数，不改变结果

In [25]:
logits.max(dim=-1)

torch.return_types.max(
values=tensor([10.7865, 12.2558, 17.3982, 13.2739, 10.6965, 10.7865,  9.5145,  9.0495,
        14.0280, 11.8378,  9.9038, 15.4187, 10.7865, 10.1476,  9.8372, 11.7660,
        10.7865, 10.0029,  9.2940,  9.6824, 11.4241,  9.4885,  8.1164,  9.5176,
        12.6383, 10.7865, 10.6021, 11.0822,  6.3617, 17.3157, 12.4544,  8.1669],
       grad_fn=<MaxBackward0>),
indices=tensor([ 1,  8,  9,  0, 15,  1, 17,  2,  9,  9,  2,  0,  1, 15,  1,  0,  1, 19,
         1,  1, 16, 10, 26,  9,  0,  1, 15, 16,  3,  9, 19,  1]))

In [104]:
# 建立数据集
def build_dataset(words):
    
    block_size=3 #上下文长度，也就是模型一次能看到几个字符
    X,Y=[],[]
    for w in words: #只用前1000个单词来训练
        #print(w)
        context=[0]*block_size #用0来表示context的开始
        for ch in w+'.':
            ix=stoi[ch]
            X.append(context)
            Y.append(ix)
            #print(''.join(itos[i] for i in context), '---->', itos[ix])
            context=context[1:]+[ix] #把context向右移动，加入新的字符


    X=torch.tensor(X)
    Y=torch.tensor(Y)
    print(X.shape, Y.shape)
    return X,Y

import random
random.seed(42)
random.shuffle(words) #打乱数据集
n1=int(0.8*len(words))
n2=int(0.9*len(words))

Xtr,Ytr=build_dataset(words[:n1])
Xdev,Ydev=build_dataset(words[n1:n2])
Xte,Yte=build_dataset(words[n2:])
    

torch.Size([182580, 3]) torch.Size([182580])
torch.Size([22767, 3]) torch.Size([22767])
torch.Size([22799, 3]) torch.Size([22799])


In [105]:
## -------------------  完整步骤(把数据集划分后的) -------------------
Xtr.shape, Xtr.dtype, Ytr.shape, Ytr.dtype

(torch.Size([182580, 3]), torch.int64, torch.Size([182580]), torch.int64)

In [112]:
g=torch.Generator().manual_seed(2147483647)
C=torch.randn(27,2, generator=g)
w1=torch.randn((6,100), generator=g)
b1=torch.randn(100, generator=g)
w2=torch.randn((100,27), generator=g)
b2=torch.randn(27, generator=g)
parameters=[C,w1,b1,w2,b2]

In [113]:
for p in parameters:
    p.requires_grad=True

In [118]:
# lri=[]
# lossi=[]
for i in range(1000):

    #构建 mini batch
    ix=torch.randint(0,Xtr.shape[0],(32,))
    # forward pass
    emb=C[X[ix]]  #(32,3,2)
    h=torch.tanh(emb.view(-1,6) @ w1 + b1) #(32,100)
    logits=h @ w2 + b2 #(32,27)
    # counts=logits.exp()
    # prob=counts/counts.sum(1,keepdim=True)
    # loss=-prob[torch.arange(32),Y].log().mean()
    loss=F.cross_entropy(logits, Ytr[ix])
    

    #backward pass
    for p in parameters:
        p.grad=None
    loss.backward()
    #update
    # lr=lrs[i]
    lr=0.1
    for p in parameters:
        p.data +=  -lr* p.grad
    #track stats
    # lri.append(lr)
    # lossi.append(loss.item())

print(loss.item())

2.788922071456909


In [ ]:
emb=C[Xdev]
h=torch.tanh(emb.view(-1,6) @ w1 + b1)
logits=h @ w2 + b2
loss=F.cross_entropy(logits, Ydev)
print(loss)